# CSCI 447/547 Hackathon 4: Naïve Bayes

This notebook is designed to be started during class and continued as a take-home activity

## How to use hackathon notebooks:

If the topic covered in a hackathon is new to you, work through the cells in order and read the explanation before running each code cell. You do not need to understand every detail of the hackathon on the first pass, instead, focus on the approach we take:

1. **Look at the data**
2. **Separate inputs from the target we want to predict**
3. **Split the data so we can test whether the model generalizes**
4. **Fit a model using the training data**
5. **Make predictions and evaluate them**
6. **Improve the model carefully <u>without</u> using the final evaluation data to make decisions**

We will work through the salary example together as a class. Pause before important code cells and think about what you expect to see. Please ask Lucy or a TA questions any time a term or line of code is unfamiliar.

After class, continue from wherever you left off. The existing explanations and code will be there to guide you through the process. Complete the marked answer sections, run every cell, and explain what the results mean in language that makes sense for you. Hackathons will not be graded, they are only to help you, and you will get out of them what you put into them.

<h4><span style="color:red">The goal of this notebook is NOT to memorize every function. It is to identify the processes we use in machine learning and be able to reuse them.</span></h4>

##### In this exercise, you will implement naïve Bayes and try to classify spam and ham emails.

---

### Google Colab Instructions

If you are using Google Colab <a href="https://colab.research.google.com/github/lucywowen/csci547_ML/blob/main/examples/Linear_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Google Colab"/></a> you will need to download [spam.csv from GitHub](https://github.com/lucywowen/csci547_ML/tree/main/examples/data) and upload it to Colab for this to work.

---

### Local (VSCode/Jupyter Lab) Version

Make sure you

```bash
git pull
```

for the latest version of the repository and make sure that your `uv` virtual environment is enabled.

---

## 1. The Math Under the Hood: Bayes' Theorem

Today, we are predicting discrete categories (classification) to identify spam emails from legitimate ones.

To do this, we use **Bayes' Theorem**, a fundamental principle of conditional probability. It calculates the probability of an event occurring based on prior knowledge of conditions related to the event.

$$ P(A|B) = \frac{P(B|A) \cdot P(A)}{P(B)} $$

* **$P(A|B)$ (Posterior):** The probability of our hypothesis ($A$) being true given the evidence ($B$). *(e.g., The probability an email is spam given it contains the word "winner")*.

* **$P(B|A)$ (Likelihood):** The probability of seeing the evidence given the hypothesis is true. *(e.g., How often the word "winner" appears in known spam emails)*.

* **$P(A)$ (Prior):** The probability of the hypothesis before seeing any evidence. *(e.g., What percentage of all emails are spam?)*.

* **$P(B)$ (Evidence):** The total probability of seeing the evidence under any circumstance *(e.g., What percentage of all emails does the word "winner" appear in?)*.

---

### Think-Pair-Share 1: The Base Rate Fallacy
**Context:** Suppose you are developing a test for a rare disease. The test is 99% accurate (it correctly identifies 99% of sick people, and correctly clears 99% of healthy people). The disease is rare: only 0.2% (0.002) of the population has it. 

1. **Think:** If a randomly chosen person tests positive, is the probability they actually have the disease 99%? Why or why not? Use the concept of the "Prior" ($P(A)$) to guide your reasoning.

    *Your individual hypothesis:* **ANSWER**

2. **Pair:** Discuss with your group. (Hint: Out of 1,000 people, how many actually have the disease? How many healthy people will get a false positive?)

    *Group consensus:* **ANSWER**

3. **Share:** Be prepared to share your group's conclusion with the class.

---

## 2. Why is it called *Naïve* Bayes?

If an email has 100 words, calculating the exact probability of seeing that specific combination of 100 words in a spam email is computationally impossible. The dataset would need to be infinitely large.

To solve this, we make a **naïve assumption**: We assume every word's appearance is *statistically independent* of every other word. 

Instead of calculating $P(Word_1 \text{ and } Word_2 | Spam)$, we calculate:
$$ P(Word_1 | Spam) \times P(Word_2 | Spam) $$

While grammatically false (the word "credit" is highly dependent on the word "card"), this simplification makes the math lightning-fast and, surprisingly, highly effective for text classification.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt


In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    
    df = pd.read_csv('data.txt', encoding='latin-1') # colab file import
    
except: 
    
    data = pd.read_csv('data/spam.csv', encoding='latin-1') # local file import


### Think-Pair-Share 2: Converting Text to Math
**Context:** Machine learning algorithms like Naïve Bayes require numerical matrices to compute probabilities. However, our dataset consists of raw text strings (emails).

1. **Think:** How would you convert an email's text into an array of numbers so a mathematical model can process it? What information is inevitably lost during this conversion?

    *Your individual hypothesis:* **ANSWER**

2. **Pair:** Discuss your ideas with your group. Consider the concept of a "Bag of Words."

    *Group consensus:* **ANSWER**

3. **Share:** Be prepared to share your group's conclusion with the class.

---

### 3. Data Preprocessing & Vectorization

To feed text into our model, we use a technique called **Count Vectorization** (often referred to as a "Bag of Words" model). 

1. The algorithm builds a vocabulary dictionary of every unique word in the entire dataset.

2. It converts each email into a numerical vector representing the *frequency* (count) of each word.

*(Note: We drop empty/NaN columns (since they don't give us any useful information) and rename our features (for ease of use) before vectorizing).*

In [ ]:
# Drop the empty/NaN columns
data = data.drop(columns=['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'])

# Rename columns for clarity:
data.columns = ['label', 'text']

# Display the first 5 rows to verify the cleanup
data.head()

In [ ]:
# Separate features (X) and target labels (y)
X =  data.drop('label', axis=1)
y = data['label']

# 80-20 train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Create a CountVectorizer instance
vectorizer = CountVectorizer()

# Fit and transform the training data (X_train)
X_train_vectorized = vectorizer.fit_transform(X_train['text'])

# Transform the test data (X_test)
X_test_vectorized = vectorizer.transform(X_test['text'])

In [ ]:
# Train the Multinomial Naive Bayes classifier
classifier = MultinomialNB()
classifier.fit(X_train_vectorized, y_train)

In [ ]:
# Make predictions on the test data
y_pred = classifier.predict(X_test_vectorized)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
classification_rep = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy:.2f}")
print("Confusion Matrix:")
print(conf_matrix)
print("Classification Report:")
print(classification_rep)

In [ ]:
# Count the number of spam and non-spam emails in the test set
spam_counts = y_test.value_counts()

# Plot the histogram
plt.figure(figsize=(8, 6))
plt.bar(spam_counts.index, spam_counts.values, color=['green', 'red'])
plt.xlabel('Email Type')
plt.ylabel('Number of Emails')
plt.title('Number of Spam and Non-Spam Emails')
plt.xticks([0, 1], ['ham (Non-Spam)', 'spam'])
plt.show()

### Think-Pair-Share 3: False Positives vs. False Negatives
**Context:** Look at the Confusion Matrix and Classification Report printed above. Our model is not perfect; it made some mistakes.
* **False Positive:** A legitimate email (ham) was incorrectly flagged as spam.
* **False Negative:** A spam email successfully sneaked into the user's main inbox.

1. **Think:** From a user-experience perspective, which of these two errors is more dangerous/costly in a spam filter? Therefore, should we optimize our model for higher **Precision** or higher **Recall** on the "spam" class?
    
    *Your individual hypothesis:* **ANSWER**

2. **Pair:** Discuss the trade-offs with your group. 
    
    *Group consensus:* **ANSWER**

3. **Share:** Be prepared to share your group's conclusion with the class.

---

## 4. Coding Exercise: Build an Inference Function

Now that you have a trained model, your task is to write a Python function that takes a raw string from a user and outputs the model's prediction.

### Task Requirements:

1. **Goal:** Complete the function `classify_message(model, vectorizer, message)` by calling the correct `scikit-learn` methods.
2. **Inputs:**
    * `model`: Your trained `MultinomialNB` classifier.
    * `vectorizer`: Your fitted `CountVectorizer` object.
    * `message`: A raw Python string (e.g., "WINNER!! Claim your prize").
3. **What you need to write:**
    * Call the `transform()` method on the vectorizer.
    * Call the `predict()` method on the model.
    * Call the `predict_proba()` method on the model.
4. **What is handled for you:**
    * `scikit-learn` methods expect and return multi-dimensional Numpy arrays. To save you from fighting with array indexing, the skeleton code below extracts the final strings and floats from the arrays you generate.

In [ ]:
def classify_message(model, vectorizer, message):
    """
    Classifies a raw string message as spam or ham using a trained Naive Bayes model.
    model: The trained Naive Bayes model.
    vectorizer: The CountVectorizer used to transform the training data.
    message: The raw string message to classify.
    """
    # 1. Transform the raw string into a numeric vector.
    # CAUTION: The vectorizer expects an iterable, so pass '[foo]' instead of just 'foo'.
    vectorized_msg = # --- YOUR CODE HERE ---
    
    
    # 2. Get the discrete prediction (spam or ham).
    # Pass into the model's prediction method.
    prediction_array = # --- YOUR CODE HERE ---
    
    
    # (Extracts the actual string from the returned numpy array)
    prediction_string = prediction_array[0] 
    
    
    # 3. Get the probability of the prediction.
    # Pass into the model's probability method.
    probability_array = # --- YOUR CODE HERE ---
    
    
    # (Extracts the highest probability float from the 2D array and converts to a percentage)
    confidence = probability_array[0].max() * 100 
    
    
    # 4. Print the results (Handled for you!)
    print(f"Message: '{message}'")
    print(f"Prediction: {prediction_string.upper()} (Confidence: {confidence:.2f}%)\n")


# --- Test your function on these two messages: ---

msg_1 = "WINNER!! This is the secret code to unlock the money: C3421."
msg_2 = "Sounds good, Tom, then see u there"

# Uncomment these lines to test your code!
# classify_message(classifier, vectorizer, msg_1)
# classify_message(classifier, vectorizer, msg_2)

---

## Hackathon Takeaways & Synthesis

As your group finishes this notebook, collaborate to answer the following synthesis questions. 

1. **Trade-off Analysis:** 
Modern Large Language Models (LLMs) like ChatGPT process text by understanding the deep contextual relationship between words. Naïve Bayes completely ignores word order and context. Given this, what is the computational advantage of continuing to use Naïve Bayes for simple text classification tasks?

    **ANSWER**

2. **Diagnosing Failures:** 
Suppose your spam filter starts incorrectly flagging almost every email as spam. Upon inspection, you realize the model learned that the word "the" is highly predictive of spam, simply because spam emails in your dataset happen to be very long. What preprocessing step could you add to your `CountVectorizer` to fix this?

    **ANSWER**

3. **3-2-1 Summary:**
* **3** key mathematical or programming concepts your group solidified today:

    1. 

    2. 

    3. 

* **2** things you are still slightly confused about (we will address these in the next lecture!):

    1. 

    2. 

* **1** real-world scenario (other than spam filtering) where you would apply Naïve Bayes classification:
    
    1.